# Vision Assistant — fine-tune YOLO11n on the merged dataset (run `n640_v1`)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Upload `training/data/merged.zip`
(made with `cd training/data && zip -qr merged.zip merged -x "*.cache"`) to Google Drive at `MyDrive/vision/merged.zip`.

Fine-tuning = keep the COCO-pretrained network, swap the output head for our 15 classes, keep training with a
small learning rate. Everything we save goes to `MyDrive/vision/` so a Colab disconnect does not lose the run.

In [ ]:
# Cell 1 — setup: install, mount Drive, unzip the dataset to local disk (much faster than reading from Drive)
!pip -q install ultralytics onnx onnxslim onnxruntime   # onnxslim = graph simplifier used by export(simplify=True)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/data && mkdir -p /content/data && unzip -q /content/drive/MyDrive/vision/merged.zip -d /content/data
import yaml, json, pathlib
y = yaml.safe_load(open('/content/data/merged/data.yaml'))
y['path'] = '/content/data/merged'          # data.yaml was written on the Mac with an absolute Mac path
yaml.safe_dump(y, open('/content/data/merged/data.yaml', 'w'), sort_keys=False)
classes = y['names']
RUNS = '/content/drive/MyDrive/vision/runs'
print(len(classes), classes)

In [ ]:
# Cell 2 — dataset figure for the report: instances per class per split
import matplotlib.pyplot as plt
counts = json.load(open('/content/data/merged/counts.json'))
plt.figure(figsize=(11, 4))
for i, s in enumerate(['train', 'val', 'test']):
    plt.bar([x + i * 0.27 for x in range(len(classes))], [counts[s].get(c, 0) for c in classes], width=0.27, label=s)
plt.xticks([x + 0.27 for x in range(len(classes))], classes, rotation=60)
plt.yscale('log'); plt.legend(); plt.title('instances per class per split (log scale)'); plt.tight_layout()
plt.savefig('/content/drive/MyDrive/vision/fig_class_counts.png', dpi=150)
plt.show()

In [ ]:
# Cell 3 — train run 1. ~1 h on a T4. Checkpoints go to Drive every epoch (last.pt / best.pt).
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/data/merged/data.yaml', epochs=50, imgsz=640, batch=32, patience=10,
            project=RUNS, name='n640_v1', exist_ok=True, seed=0)

**If Colab disconnected mid-training**, re-run Cell 1 and then this cell instead of Cell 3 — it continues from the
last saved epoch. Otherwise skip it.

In [ ]:
# Cell 3b — resume only if the run was interrupted
from ultralytics import YOLO
YOLO(f'{RUNS}/n640_v1/weights/last.pt').train(resume=True)

In [ ]:
# Cell 4 — evaluate best.pt on the untouched TEST split and save the numbers
from ultralytics import YOLO
best = YOLO(f'{RUNS}/n640_v1/weights/best.pt')
m = best.val(data='/content/data/merged/data.yaml', split='test', imgsz=640, project=RUNS, name='n640_v1_test', exist_ok=True)
per_class = {classes[c]: {'map50': float(ap50), 'map50_95': float(ap)}
             for c, ap50, ap in zip(m.box.ap_class_index, m.box.ap50, m.box.ap)}
print(f'TEST  mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}   (baseline on Mac: 0.496 on 8 COCO classes, 0 on the rest)')
for name, v in per_class.items():
    flag = '  <-- weak, see Task 9' if v['map50'] < 0.2 else ''
    print(f'{name:12s} mAP50 {v["map50"]:.3f}   mAP50-95 {v["map50_95"]:.3f}{flag}')
epochs_trained = sum(1 for _ in open(f'{RUNS}/n640_v1/results.csv')) - 1   # minus header line
json.dump({'run': 'n640_v1', 'epochs_trained': epochs_trained,
           'map50': float(m.box.map50), 'map': float(m.box.map), 'per_class': per_class},
          open(f'{RUNS}/n640_v1_test.json', 'w'), indent=2)

In [ ]:
# Cell 5 — export ONNX for the browser (fixed 1x3x640x640 input, output 1x(4+nc)x8400) and copy to Drive
onnx_path = best.export(format='onnx', opset=17, imgsz=640, simplify=True, dynamic=False)
import shutil
shutil.copy(onnx_path, '/content/drive/MyDrive/vision/best_v1.onnx')
json.dump(classes, open('/content/drive/MyDrive/vision/labels.json', 'w'))
print('saved', onnx_path, '->', 'MyDrive/vision/best_v1.onnx and labels.json')

## Bring home (Task 4 step 4)
From `MyDrive/vision/`:
- `best_v1.onnx` → `public/models/best.onnx`, `labels.json` → `public/models/labels.json`
- `runs/n640_v1/results.csv`, `results.png`, `confusion_matrix.png`, `runs/n640_v1_test.json`, `fig_class_counts.png` → `training/results/n640_v1/`
- File → Download .ipynb → `training/train.ipynb`